# dataset.csv ETL Pipeline

**Extract → Transform → Load** notebook for the indoor Wi-Fi fingerprinting data (`dataset.csv`).

## Data Structure (67 columns, 1,539 rows, `;`-delimited)
| Group | # Columns | Contents |
|---|---|---|
| Metadata | 13 | `measId`, `measTimestamp`, `Position X/Y/Z`, `zoneId`, `Zonename`, `meas X/Y/Z`, `gps*` |
| AP (RSSI) | 32 | Signal strength (dBm) per AP name; missing values are the string `null` |
| MAC flags | 22 | Detection flags (0/1) for MAC addresses (and device-name+MAC pairs) |

## Processing Summary
1. **Extract**: load `;`-delimited CSV, convert the string `null` → NaN
2. **Transform**: parse timestamps, numeric conversion, drop all-null GPS columns, quality checks, fill missing RSSI with -100, derived columns (date/hour)
3. **Load**: save `output/dataset_clean.csv` and `output/dataset_clean.parquet`, then reload to verify

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path(r"C:\Users\USER\OneDrive\Desktop\[20.08.26] Light rain\02.hybrid_dataset\dataset.csv")
OUT_DIR = RAW_PATH.parent / "output"
OUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")

print("RAW_PATH:", RAW_PATH)
print("RAW_PATH exists:", RAW_PATH.exists())

RAW_PATH: C:\Users\USER\OneDrive\Desktop\[20.08.26] Light rain\02.hybrid_dataset\dataset.csv
RAW_PATH exists: True


## 1. Extract

Read the CSV, treating the string `null` and empty strings as NaN.

In [2]:
df_raw = pd.read_csv(
    RAW_PATH,
    sep=";",
    quotechar='"',
    na_values=["null", ""],
    keep_default_na=True,
    encoding="utf-8",
)

print("shape:", df_raw.shape)
df_raw.head(3)

shape: (1539, 67)


,measId,measTimestamp,Position X,Position Y,Position Z,zoneId,Zonename,meas X,meas Y,meas Z,gpsLatitude,gpsLongitude,gpsAltitude,N,109.0,AIT-L15,aut-sams-1,bolyai_E4_floor3,Bosch_Telemetry,dd,doa2,doa200,doa203,doa207,doa208,doa6,EET_3,FRM,GEIAKFSZ,IITAP1,IITAP1-GUEST,IITAP2,IITAP2-GUEST,IITAP3,IITAP3-GUEST,info,info2,KEMA10,kemA4,KRZ,library114,TP-LINK_B2765A,UPC Wi-Free,UPC8902044,wireless,00:16:53:4C:B1:F9,00:16:53:4C:B2:02,00:16:53:4C:B4:EB,00:16:53:4C:E9:1D,00:16:53:4C:F2:6A,00:16:53:4C:F5:2D,00:16:53:4C:F9:A4,00:16:53:4C:FA:60,00:16:53:4C:FA:67,48:5A:B6:54:35:DC,6B:C2:26:12:62:60,DANI 6B:C2:26:12:62:60,DM06082 48:5A:B6:54:35:DC,EV3 00:16:53:4C:E9:1D,EV3 00:16:53:4C:F2:6A,EV3 00:16:53:4C:FA:60,EV3 00:16:53:4C:FA:67,EV3BD 00:16:53:4C:F5:2D,IZE 00:16:53:4C:B1:F9,JOE 00:16:53:4C:F9:A4,MEGAROBOT 00:16:53:4C:B2:02,MrEv3 00:16:53:4C:B4:EB
0,04550b4e-b5fc-4665-a043-18e82e94399a,2016-02-27 16:40:22.0,8.0,8.0,4.4,07a25de0-a013-486d-9463-404a348e05ee,1st Floor East Corridor,0.1,-0.9,0.0,NaN,NaN,NaN,NaN,-70.0,NaN,-80.0,NaN,NaN,-83.0,NaN,NaN,-82.0,-74.0,-53.0,-83.0,NaN,NaN,-63.0,-55.0,-57.0,-78.0,-79.0,-74.0,-73.0,NaN,NaN,NaN,NaN,-65.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
1,93645336-eb2d-4983-950f-cb0e4191b986,2016-02-27 16:43:13.0,12.0,9.0,4.4,07a25de0-a013-486d-9463-404a348e05ee,1st Floor East Corridor,1.3,-1.0,0.1,NaN,NaN,NaN,NaN,-69.0,NaN,-82.0,NaN,NaN,-75.0,NaN,NaN,NaN,-82.0,-68.0,NaN,NaN,NaN,-65.0,-75.0,-81.0,-79.0,-80.0,-73.0,-73.0,NaN,NaN,NaN,NaN,-72.0,NaN,-85.0,NaN,NaN,NaN,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,7f210fdb-7018-454c-978b-9a595ee39130,2016-02-27 16:50:48.0,23.0,8.0,4.4,07a25de0-a013-486d-9463-404a348e05ee,1st Floor East Corridor,-0.5,-1.0,0.1,NaN,NaN,NaN,NaN,-65.0,NaN,-79.0,NaN,NaN,-81.0,NaN,-85.0,NaN,NaN,-75.0,NaN,NaN,NaN,-79.0,-74.0,-82.0,-82.0,-81.0,-86.0,-86.0,NaN,NaN,NaN,NaN,-85.0,-86.0,-72.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0


In [3]:
missing = df_raw.isna().mean().sort_values(ascending=False)
print("Overall missing rate: {:.2f}%".format(df_raw.isna().mean().mean() * 100))
missing.head(15)

Overall missing rate: 37.02%


gpsAltitude        1.0
gpsLongitude       1.0
gpsLatitude        1.0
UPC Wi-Free        1.0
UPC8902044         1.0
kemA4              1.0
bolyai_E4_floor3   1.0
EET_3              1.0
KEMA10             1.0
info               0.9
info2              0.9
aut-sams-1         0.9
TP-LINK_B2765A     0.9
wireless           0.8
KRZ                0.8
dtype: float64

## 2. Transform

### 2.1 Column cleanup and group separation
Strip whitespace from column names, then split AP (RSSI) vs MAC flag columns based on whether the name contains `:`.

In [4]:
df_raw.columns = [c.strip() for c in df_raw.columns]

META_COLS = [
    "measId", "measTimestamp",
    "Position X", "Position Y", "Position Z",
    "zoneId", "Zonename",
    "meas X", "meas Y", "meas Z",
    "gpsLatitude", "gpsLongitude", "gpsAltitude",
]

ap_cols = [c for c in df_raw.columns if c not in META_COLS and ":" not in c]
mac_cols = [c for c in df_raw.columns if ":" in c]

print("Metadata columns:", len(META_COLS))
print("AP (RSSI) columns:", len(ap_cols))
print("MAC flag columns:", len(mac_cols))

Metadata columns: 13
AP (RSSI) columns: 32
MAC flag columns: 22


### 2.2 Type conversion

- `measTimestamp`: `2016-02-27 16:40:22.0` → datetime
- Coordinates/measurements and RSSI → numeric
- MAC flags → 0/1 (int8)

In [5]:
df = df_raw.copy()

df["measTimestamp"] = pd.to_datetime(
    df["measTimestamp"].astype(str).str.replace(r"\.0$", "", regex=True),
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce",
)

num_cols = ["Position X", "Position Y", "Position Z", "meas X", "meas Y", "meas Z"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

for c in ap_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

for c in mac_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(np.int8)

print(df.dtypes.value_counts())

float64           41
int8              22
object             3
datetime64[ns]     1
Name: count, dtype: int64


### 2.3 Drop GPS columns

`gpsLatitude` / `gpsLongitude` / `gpsAltitude` are null in all 1,539 rows → drop them since they carry no information.

In [6]:
gps_cols = ["gpsLatitude", "gpsLongitude", "gpsAltitude"]
print("GPS column missing counts:", df[gps_cols].isna().sum().to_dict())

df = df.drop(columns=gps_cols)
META_COLS = [c for c in META_COLS if c not in gps_cols]
print("Shape after dropping GPS:", df.shape)

GPS column missing counts: {'gpsLatitude': 1539, 'gpsLongitude': 1539, 'gpsAltitude': 1539}
Shape after dropping GPS: (1539, 64)


### 2.4 Data quality checks

In [7]:
checks = {
    "Total rows": len(df),
    "Unique measId": df["measId"].nunique(),
    "Duplicate measId": int(df["measId"].duplicated().sum()),
    "Missing timestamps": int(df["measTimestamp"].isna().sum()),
    "Timestamp range": f"{df['measTimestamp'].min()} ~ {df['measTimestamp'].max()}",
    "Unique Zonename": df["Zonename"].nunique(),
    "RSSI min/max": (df[ap_cols].min().min(), df[ap_cols].max().max()),
    "RSSI cells outside -100..0": int(((df[ap_cols] < -100) | (df[ap_cols] > 0)).sum().sum()),
    "Missing Position values": int(df[["Position X", "Position Y", "Position Z"]].isna().sum().sum()),
    "Missing Zonename": int(df["Zonename"].isna().sum()),
}

for k, v in checks.items():
    print(f"{k}: {v}")

Total rows: 1539
Unique measId: 1539
Duplicate measId: 0
Missing timestamps: 0
Timestamp range: 2016-02-27 08:49:23 ~ 2016-02-27 19:51:47
Unique Zonename: 21
RSSI min/max: (-96.0, -34.0)
RSSI cells outside -100..0: 0
Missing Position values: 0
Missing Zonename: 0


### 2.5 Missing-value handling and derived columns

- Missing RSSI → **-100** (no signal; common Wi-Fi fingerprinting convention)
- Missing MAC flags were already filled with 0 in 2.2
- Derived columns: `date`, `hour`
- Normalize whitespace in `Zonename`

In [8]:
df = df.sort_values("measTimestamp").reset_index(drop=True)

df[ap_cols] = df[ap_cols].fillna(-100)

df["date"] = df["measTimestamp"].dt.date
df["hour"] = df["measTimestamp"].dt.hour
df["Zonename"] = df["Zonename"].astype(str).str.strip()

print("Shape after transform:", df.shape)
print("Missing RSSI values:", int(df[ap_cols].isna().sum().sum()))
df.head(3)

Shape after transform: (1539, 66)
Missing RSSI values: 0


,measId,measTimestamp,Position X,Position Y,Position Z,zoneId,Zonename,meas X,meas Y,meas Z,N,109.0,AIT-L15,aut-sams-1,bolyai_E4_floor3,Bosch_Telemetry,dd,doa2,doa200,doa203,doa207,doa208,doa6,EET_3,FRM,GEIAKFSZ,IITAP1,IITAP1-GUEST,IITAP2,IITAP2-GUEST,IITAP3,IITAP3-GUEST,info,info2,KEMA10,kemA4,KRZ,library114,TP-LINK_B2765A,UPC Wi-Free,UPC8902044,wireless,00:16:53:4C:B1:F9,00:16:53:4C:B2:02,00:16:53:4C:B4:EB,00:16:53:4C:E9:1D,00:16:53:4C:F2:6A,00:16:53:4C:F5:2D,00:16:53:4C:F9:A4,00:16:53:4C:FA:60,00:16:53:4C:FA:67,48:5A:B6:54:35:DC,6B:C2:26:12:62:60,DANI 6B:C2:26:12:62:60,DM06082 48:5A:B6:54:35:DC,EV3 00:16:53:4C:E9:1D,EV3 00:16:53:4C:F2:6A,EV3 00:16:53:4C:FA:60,EV3 00:16:53:4C:FA:67,EV3BD 00:16:53:4C:F5:2D,IZE 00:16:53:4C:B1:F9,JOE 00:16:53:4C:F9:A4,MEGAROBOT 00:16:53:4C:B2:02,MrEv3 00:16:53:4C:B4:EB,date,hour
0,3e8822a5-fedf-45ce-9295-bd4ffbc51c74,2016-02-27 08:49:23,2.0,7.0,1.5,1501dc2f-55e3-44bd-8f15-8c26a8c7410d,Ground Floor East Corridor,-0.5,-0.7,0.1,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-83.0,-100.0,-100.0,-100.0,-83.0,-77.0,-100.0,-100.0,-64.0,-80.0,-79.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-77.0,-100.0,-100.0,-100.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2016-02-27,8
1,8444898e-b89f-4dea-a881-ece3d6ff865e,2016-02-27 08:50:33,4.0,7.0,1.5,1501dc2f-55e3-44bd-8f15-8c26a8c7410d,Ground Floor East Corridor,-0.4,-0.9,0.2,-100.0,-89.0,-100.0,-100.0,-100.0,-100.0,-100.0,-86.0,-100.0,-100.0,-100.0,-81.0,-77.0,-100.0,-100.0,-51.0,-74.0,-72.0,-100.0,-100.0,-90.0,-88.0,-100.0,-100.0,-100.0,-100.0,-89.0,-100.0,-85.0,-100.0,-100.0,-100.0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,2016-02-27,8
2,506e5b69-4f5f-48c1-abba-689d0769072d,2016-02-27 08:51:08,5.0,7.0,1.5,1501dc2f-55e3-44bd-8f15-8c26a8c7410d,Ground Floor East Corridor,-2.3,-0.9,0.3,-100.0,-85.0,-100.0,-100.0,-100.0,-100.0,-100.0,-85.0,-100.0,-100.0,-90.0,-81.0,-70.0,-100.0,-100.0,-51.0,-78.0,-78.0,-100.0,-100.0,-100.0,-89.0,-100.0,-100.0,-100.0,-100.0,-86.0,-100.0,-74.0,-100.0,-100.0,-100.0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,2016-02-27,8


## 3. Load

Save the cleaned result to `output/` as CSV and Parquet.

In [9]:
CLEAN_CSV = OUT_DIR / "dataset_clean.csv"
PARQUET = OUT_DIR / "dataset_clean.parquet"

df.to_csv(CLEAN_CSV, sep=";", index=False, encoding="utf-8")

try:
    df.to_parquet(PARQUET, index=False)
    print("Parquet saved:", PARQUET)
except Exception as e:
    print("Parquet save failed (pyarrow not installed):", e)

print("CSV saved:", CLEAN_CSV, f"({CLEAN_CSV.stat().st_size / 1e6:.2f} MB)")

Parquet saved: C:\Users\USER\OneDrive\Desktop\[20.08.26] Light rain\02.hybrid_dataset\output\dataset_clean.parquet


CSV saved: C:\Users\USER\OneDrive\Desktop\[20.08.26] Light rain\02.hybrid_dataset\output\dataset_clean.csv (0.70 MB)


### 3.1 Reload verification

In [10]:
df_check = pd.read_csv(
    CLEAN_CSV,
    sep=";",
    na_values=["null", ""],
    parse_dates=["measTimestamp"],
)

mac_vals = sorted(pd.unique(df_check[mac_cols].values.ravel()))

print("Reloaded shape:", df_check.shape)
print("measTimestamp dtype:", df_check["measTimestamp"].dtype)
print("Missing RSSI values:", int(df_check[ap_cols].isna().sum().sum()))
print("MAC flag unique values:", mac_vals)
print("Unique Zonename:", df_check["Zonename"].nunique())
df_check[["measId", "measTimestamp", "Zonename", "Position X", "Position Y", "hour"]].head()

Reloaded shape: (1539, 66)
measTimestamp dtype: datetime64[ns]
Missing RSSI values: 0
MAC flag unique values: [np.int64(0), np.int64(1)]
Unique Zonename: 21


,measId,measTimestamp,Zonename,Position X,Position Y,hour
0,3e8822a5-fedf-45ce-9295-bd4ffbc51c74,2016-02-27 08:49:23,Ground Floor East Corridor,2.0,7.0,8
1,8444898e-b89f-4dea-a881-ece3d6ff865e,2016-02-27 08:50:33,Ground Floor East Corridor,4.0,7.0,8
2,506e5b69-4f5f-48c1-abba-689d0769072d,2016-02-27 08:51:08,Ground Floor East Corridor,5.0,7.0,8
3,6f3452b0-2145-4d34-87bc-100c6f1eeac8,2016-02-27 08:52:05,Ground Floor East Corridor,6.0,7.0,8
4,79c688a9-d8ce-454b-b92b-80347cc2c725,2016-02-27 08:54:14,Ground Floor East Corridor,8.0,7.0,8


## Summary

| Step | Details |
|---|---|
| Extract | `;`-delimited CSV, 1,539 rows × 67 columns, `null` → NaN |
| Transform | datetime parsing, numeric conversion, GPS (all null) removed → 64 columns, missing RSSI → -100, date/hour derived columns |
| Load | `output/dataset_clean.csv` / `output/dataset_clean.parquet` |

Author : Judit Tams, Zsolt Tth
